# Full-network corrected allocation benchmark

This notebook runs seeds 42, 43, and 44 at matched global sparsities of 50% and 75%.

It compares:

- additive RDP allocation;
- pairwise-interaction-corrected RDP allocation;
- direct empirical-distortion RDP allocation;
- reconstructed RDHP allocation;
- dense baseline.

For each method it records calibration/test logit distortion, test distortion regret, retained parameters, approximate FLOPs, immediate accuracy, and 5/20-epoch fine-tuned accuracy.

The pair-corrected allocator is an explicit heuristic: it measures a fixed set of pair interactions at 50% local pruning, distributes each interaction correction across the two layers, and feeds the corrected curves into the same global dynamic program. This makes the comparison reproducible without claiming that it is the exact paper algorithm.

## Requirements

- Colab GPU runtime.
- CIFAR-10 at `/content/drive/MyDrive/cifar10/cifar-10-batches-py`.
- Dense checkpoints at `/content/drive/MyDrive/rdhp_rdp_runs_seed_{42,43,44}/dense_resnet32.pt`.
- Secret `GITHUB_TOKEN` with read access to private `my-thesis` and its submodules.
- Secret `GITHUB_WRITE_TOKEN` with Contents read/write access to this experiment repository.

This run is substantially heavier than the additivity notebook because it builds full-network curves and fine-tunes multiple masks.

In [ ]:
from pathlib import Path
import copy
import getpass
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
from argparse import Namespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from google.colab import drive

OWNER = 'hamidrg20001379'
REPO_NAME = 'my-thesis'
REPO_URL = f'https://github.com/{OWNER}/{REPO_NAME}.git'
REPO = Path('/content') / REPO_NAME

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass
if not token:
    token = getpass.getpass('GitHub token with private-repository read access: ')
if not token:
    raise RuntimeError('A GitHub token is required.')

if REPO.exists() and not (REPO / '.git').exists():
    raise RuntimeError(f'{REPO} exists but is not a Git checkout.')

askpass = Path('/tmp/github-askpass-full-network.sh')
askpass.write_text(
    '#!/bin/sh\n'
    'case "$1" in\n'
    "  *Username*) printf '%s\\n' x-access-token ;;\n"
    "  *Password*) printf '%s\\n' \"$GITHUB_TOKEN_FOR_ASKPASS\" ;;\n"
    "  *) printf '\\n' ;;\n"
    'esac\n'
)
askpass.chmod(0o700)
git_env = os.environ.copy()
git_env.update({
    'GIT_ASKPASS': str(askpass),
    'GIT_TERMINAL_PROMPT': '0',
    'GITHUB_TOKEN_FOR_ASKPASS': token,
})
try:
    if not REPO.exists():
        subprocess.run(['git', 'clone', '--recurse-submodules', REPO_URL, str(REPO)],
                       env=git_env, check=True)
    else:
        subprocess.run(['git', '-C', str(REPO), 'submodule', 'update', '--init', '--recursive'],
                       env=git_env, check=True)
finally:
    askpass.unlink(missing_ok=True)

EXPERIMENTS = REPO / 'working' / 'experiments'
sys.path.insert(0, str(EXPERIMENTS))

from rdp_resnet32_adapter import (
    accuracy,
    allocate_dp,
    apply_masks,
    build_curves as build_rdp_curves,
    load_checkpoint,
    load_resnet32,
    logits,
    make_loaders,
    model_stats,
    prunable_layers,
    seed_everything,
    train_epochs,
)
from rdhp_resnet32_adapter import build_rdhp_curves

DRIVE_ROOT = Path('/content/drive/MyDrive')
if not DRIVE_ROOT.exists():
    drive.mount('/content/drive')

DATA_ROOT = DRIVE_ROOT / 'cifar10'
DATASET_DIR = DATA_ROOT / 'cifar-10-batches-py'
for name in ['data_batch_1', 'data_batch_2', 'data_batch_3', 'data_batch_4', 'data_batch_5', 'test_batch']:
    if not (DATASET_DIR / name).exists():
        raise FileNotFoundError(f'Missing CIFAR-10 file: {DATASET_DIR / name}')

SEEDS = [42, 43, 44]
TARGET_SPARSITIES = [0.50, 0.75]
TRAIN_SAMPLES = 5000
CALIB_SAMPLES = 1024
TEST_SAMPLES = 1000
BATCH_SIZE = 128
CURVE_POINTS = 20
BUDGET_BINS = 5000
FISHER_PROBES = 2
FISHER_DAMPING = 1e-7
PAIR_COUNT = 24
PAIR_REFERENCE_RATIO = 0.50
FINETUNE_EPOCHS = [5, 20]
RUN_FINETUNE = True
LEARNING_RATE = 0.1
PUBLISH_RESULTS = True

OUTPUT_ROOT = DRIVE_ROOT / 'rdhp_rdp_full_network_corrected_all_seeds'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('Seeds:', SEEDS)
print('Target sparsities:', TARGET_SPARSITIES)
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('Output:', OUTPUT_ROOT)

In [ ]:
probe = load_resnet32()
layer_names = [
    name for name, module in probe.named_modules()
    if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear))
]
layer_count = len(layer_names)
del probe

rng = np.random.default_rng(123)
all_pairs = [(a, b) for a in range(layer_count) for b in range(a + 1, layer_count)]
pair_indices = [all_pairs[i] for i in rng.choice(
    len(all_pairs), size=min(PAIR_COUNT, len(all_pairs)), replace=False
)]

print('Prunable layers:', layer_count)
print('Pair correction pairs:', pair_indices)
print('Pair names:')
for a, b in pair_indices:
    print((a, layer_names[a]), (b, layer_names[b]))

In [ ]:
def all_one_masks(model):
    return [
        torch.ones_like(layer.weight, dtype=torch.bool).cpu()
        for layer in prunable_layers(model)
    ]

def masks_at_ratios(model, ratios, selected_ratios):
    masks = all_one_masks(model)
    layers = prunable_layers(model)
    for index, ratio in enumerate(selected_ratios):
        masks[index] = mask_topk(layers[index].weight, float(ratio)).detach().cpu()
    return masks

def mask_topk(weight, fraction):
    flat = weight.detach().abs().flatten()
    keep = max(1, min(flat.numel(), int(round((1.0 - float(fraction)) * flat.numel()))))
    order = torch.argsort(flat, descending=True, stable=True)
    mask = torch.zeros_like(flat, dtype=torch.bool)
    mask[order[:keep]] = True
    return mask.reshape_as(weight)

def measure_mask(model, masks, calibration_loader, test_loader,
                 dense_calib, dense_test, device):
    masked = copy.deepcopy(model).to(device)
    apply_masks(masked, masks)
    masked_calib = logits(masked, calibration_loader, device)
    masked_test = logits(masked, test_loader, device)
    calib_error = (masked_calib - dense_calib).flatten(1).square().mean(dim=1)
    test_error = (masked_test - dense_test).flatten(1).square().mean(dim=1)
    result = {
        'calib_mean_distortion': float(calib_error.mean()),
        'test_mean_distortion': float(test_error.mean()),
        'calib_max_distortion': float(calib_error.max()),
        'test_max_distortion': float(test_error.max()),
    }
    del masked, masked_calib, masked_test
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return result

def measure_calib_mean(model, masks, calibration_loader, dense_calib, device):
    masked = copy.deepcopy(model).to(device)
    apply_masks(masked, masks)
    masked_calib = logits(masked, calibration_loader, device)
    error = (masked_calib - dense_calib).flatten(1).square().mean(dim=1)
    value = float(error.mean())
    del masked, masked_calib
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return value

def estimate_flops(model, masks, loader, device):
    records = []
    handles = []

    def hook(module, inputs, output):
        if isinstance(module, nn.Conv2d):
            output_elements = int(output.numel())
            kernel_ops = (module.in_channels // module.groups) * int(np.prod(module.kernel_size))
            dense_ops = output_elements * kernel_ops
        elif isinstance(module, nn.Linear):
            dense_ops = int(output.numel()) * module.in_features
        else:
            dense_ops = 0
        active_fraction = float(masks[len(records)].float().mean())
        records.append((dense_ops, dense_ops * active_fraction))

    for layer in prunable_layers(model):
        handles.append(layer.register_forward_hook(hook))

    model.eval()
    x, _ = next(iter(loader))
    with torch.no_grad():
        model(x.to(device))
    for handle in handles:
        handle.remove()

    dense_flops = sum(x[0] for x in records)
    sparse_flops = sum(x[1] for x in records)
    return {
        'dense_flops_per_batch': float(dense_flops),
        'sparse_flops_per_batch': float(sparse_flops),
        'flops_retained_fraction': float(sparse_flops / max(dense_flops, 1.0)),
        'flops_reduction_fraction': float(1.0 - sparse_flops / max(dense_flops, 1.0)),
    }

def jsonable(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.ndarray,)):
        return value.tolist()
    if isinstance(value, (np.bool_,)):
        return bool(value)
    return value

In [ ]:
def prepare_allocations(model, calibration_loader, dense_calib, device, seed):
    started = time.monotonic()

    ratios, worst_curves = build_rdp_curves(
        model, calibration_loader, device,
        curve_points=CURVE_POINTS,
        max_pruning=1.0,
        worst_case=True,
        monotonic_repair=True,
    )
    mean_ratios, mean_curves = build_rdp_curves(
        model, calibration_loader, device,
        curve_points=CURVE_POINTS,
        max_pruning=1.0,
        worst_case=False,
        monotonic_repair=False,
    )
    if not np.allclose(ratios, mean_ratios):
        raise RuntimeError('RDP curve grids do not match.')

    reference_index = int(np.argmin(np.abs(ratios - PAIR_REFERENCE_RATIO)))
    pair_rows = []
    corrected_curves = mean_curves.copy()

    for layer_a, layer_b in pair_indices:
        masks_a = all_one_masks(model)
        masks_b = all_one_masks(model)
        masks_ab = all_one_masks(model)
        layers = prunable_layers(model)
        masks_a[layer_a] = mask_topk(layers[layer_a].weight, PAIR_REFERENCE_RATIO).detach().cpu()
        masks_b[layer_b] = mask_topk(layers[layer_b].weight, PAIR_REFERENCE_RATIO).detach().cpu()
        masks_ab[layer_a] = masks_a[layer_a]
        masks_ab[layer_b] = masks_b[layer_b]

        single_a = float(mean_curves[layer_a, reference_index])
        single_b = float(mean_curves[layer_b, reference_index])
        joint = measure_calib_mean(model, masks_ab, calibration_loader, dense_calib, device)
        interaction = joint - single_a - single_b
        pair_rows.append({
            'layer_a': layer_a,
            'layer_a_name': layer_names[layer_a],
            'layer_b': layer_b,
            'layer_b_name': layer_names[layer_b],
            'reference_ratio': PAIR_REFERENCE_RATIO,
            'single_a': single_a,
            'single_b': single_b,
            'joint': joint,
            'interaction': interaction,
        })

        for layer_index, contribution in [
            (layer_a, interaction / 2.0),
            (layer_b, interaction / 2.0),
        ]:
            scale = mean_curves[layer_index] / max(
                float(mean_curves[layer_index, reference_index]), 1e-12
            )
            corrected_curves[layer_index] += contribution * scale

    corrected_curves = np.maximum(corrected_curves, 0.0)

    rdhp_ratios, rdhp_curves, fishers = build_rdhp_curves(
        model, calibration_loader, device,
        curve_points=CURVE_POINTS,
        max_pruning=1.0,
        fisher_probes=FISHER_PROBES,
        fisher_damping=FISHER_DAMPING,
        fisher_mode='diagonal',
        dense_limit=2048,
        seed=seed,
    )
    if not np.allclose(ratios, rdhp_ratios):
        raise RuntimeError('RDP and RDHP curve grids do not match.')

    allocations = {}
    for target_sparsity in TARGET_SPARSITIES:
        additive_masks, additive_info = allocate_dp(
            model, ratios, worst_curves, target_sparsity, budget_bins=BUDGET_BINS
        )
        direct_masks, direct_info = allocate_dp(
            model, ratios, mean_curves, target_sparsity, budget_bins=BUDGET_BINS
        )
        corrected_masks, corrected_info = allocate_dp(
            model, ratios, corrected_curves, target_sparsity, budget_bins=BUDGET_BINS
        )
        rdhp_masks, rdhp_info = allocate_dp(
            model, ratios, rdhp_curves, target_sparsity, budget_bins=BUDGET_BINS
        )
        allocations[float(target_sparsity)] = {
            'additive_rdp': (additive_masks, additive_info),
            'pair_corrected_rdp': (corrected_masks, corrected_info),
            'direct_rdp': (direct_masks, direct_info),
            'rdhp': (rdhp_masks, rdhp_info),
        }
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    return {
        'ratios': ratios,
        'worst_curves': worst_curves,
        'mean_curves': mean_curves,
        'corrected_curves': corrected_curves,
        'rdhp_curves': rdhp_curves,
        'pair_rows': pair_rows,
        'allocations': allocations,
        'elapsed_seconds': time.monotonic() - started,
    }

In [ ]:
def run_seed(seed):
    started = time.monotonic()
    seed_everything(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    run_root = DRIVE_ROOT / f'rdhp_rdp_runs_seed_{seed}'
    checkpoint = run_root / 'dense_resnet32.pt'
    output_dir = OUTPUT_ROOT / f'seed_{seed}'
    output_dir.mkdir(parents=True, exist_ok=True)
    if not checkpoint.exists():
        raise FileNotFoundError(f'Missing checkpoint: {checkpoint}')

    loader_args = Namespace(
        synthetic=False,
        data_root=str(DATA_ROOT),
        download=False,
        train_samples=TRAIN_SAMPLES,
        calib_samples=CALIB_SAMPLES,
        test_samples=TEST_SAMPLES,
        batch_size=BATCH_SIZE,
        seed=seed,
    )
    (train_loader, calibration_loader, test_loader), split_info = make_loaders(loader_args)

    model = load_resnet32().to(device)
    checkpoint_sha256 = load_checkpoint(model, checkpoint)
    dense_calib = logits(model, calibration_loader, device)
    dense_test = logits(model, test_loader, device)
    dense_accuracy = accuracy(model, test_loader, device)
    dense_flops = estimate_flops(model, all_one_masks(model), test_loader, device)

    prepared = prepare_allocations(model, calibration_loader, dense_calib, device, seed)
    (OUTPUT_ROOT / f'seed_{seed}_pair_interactions.csv').write_text(
        pd.DataFrame(prepared['pair_rows']).to_csv(index=False)
    )

    rows = []
    curve_rows = []
    for target_sparsity, methods in prepared['allocations'].items():
        measured = {}
        for method, (masks, allocation_info) in methods.items():
            values = measure_mask(
                model, masks, calibration_loader, test_loader,
                dense_calib, dense_test, device
            )
            masked = copy.deepcopy(model).to(device)
            apply_masks(masked, masks)
            immediate_accuracy = accuracy(masked, test_loader, device)
            stats = model_stats(masked)
            flops = estimate_flops(masked, masks, test_loader, device)
            measured[method] = {
                'target_sparsity': target_sparsity,
                'method': method,
                'calib_mean_distortion': values['calib_mean_distortion'],
                'test_mean_distortion': values['test_mean_distortion'],
                'calib_max_distortion': values['calib_max_distortion'],
                'test_max_distortion': values['test_max_distortion'],
                'immediate_accuracy': immediate_accuracy,
                'dense_accuracy': dense_accuracy,
                'accuracy_loss_immediate': immediate_accuracy - dense_accuracy,
                'parameters': stats['parameters'],
                'nonzero': stats['nonzero'],
                'sparsity_actual': stats['sparsity'],
                'retained_fraction': stats['retained_fraction'],
                **flops,
                'allocation_achieved_sparsity': allocation_info['achieved_sparsity'],
            }

            if RUN_FINETUNE:
                for epochs in FINETUNE_EPOCHS:
                    finetuned = copy.deepcopy(masked).to(device)
                    train_epochs(
                        finetuned, train_loader, device, epochs, LEARNING_RATE,
                        seed + 1000 + epochs, masks=masks
                    )
                    apply_masks(finetuned, masks)
                    final_accuracy = accuracy(finetuned, test_loader, device)
                    measured[method][f'finetuned_accuracy_{epochs}'] = final_accuracy
                    measured[method][f'accuracy_loss_finetuned_{epochs}'] = final_accuracy - dense_accuracy
                    del finetuned
                    if device.type == 'cuda':
                        torch.cuda.empty_cache()

            rows.append(measured[method])
            del masked
            if device.type == 'cuda':
                torch.cuda.empty_cache()

        oracle = min(row['test_mean_distortion'] for row in measured.values())
        for row in rows[-len(measured):]:
            row['test_distortion_regret'] = (
                row['test_mean_distortion'] - oracle
            ) / max(oracle, 1e-12)
            row['test_distortion_oracle'] = oracle

        curve_rows.append({
            'seed': seed,
            'target_sparsity': target_sparsity,
            'curve_build_seconds': prepared['elapsed_seconds'],
        })
        print(
            f'Seed {seed} sparsity {target_sparsity:.2f}: '
            + ', '.join(
                f'{method} distortion={measured[method]["test_mean_distortion"]:.5f}'
                for method in measured
            )
        )

    results_df = pd.DataFrame(rows)
    results_df.to_csv(output_dir / 'method_results.csv', index=False)
    pd.DataFrame(prepared['pair_rows']).to_csv(output_dir / 'pair_interactions.csv', index=False)
    (output_dir / 'allocation_diagnostics.json').write_text(
        json.dumps({
            'seed': seed,
            'ratios': prepared['ratios'],
            'curve_build_seconds': prepared['elapsed_seconds'],
            'allocation_info': {
                str(target): {
                    method: info[1]
                    for method, info in methods.items()
                }
                for target, methods in prepared['allocations'].items()
            },
            'split': split_info,
            'checkpoint_sha256': checkpoint_sha256,
        }, indent=2, default=jsonable) + '\n'
    )
    np.savez_compressed(
        output_dir / 'allocation_curves.npz',
        ratios=prepared['ratios'],
        worst_curves=prepared['worst_curves'],
        mean_curves=prepared['mean_curves'],
        corrected_curves=prepared['corrected_curves'],
        rdhp_curves=prepared['rdhp_curves'],
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for target_sparsity in TARGET_SPARSITIES:
        plot_df = results_df[results_df['target_sparsity'] == target_sparsity]
        axes[0].plot(
            plot_df['method'], plot_df['test_mean_distortion'],
            marker='o', label=f'{target_sparsity:.0%} sparsity'
        )
    axes[0].set_ylabel('Test mean logit distortion')
    axes[0].set_title(f'Seed {seed}: distortion by method')
    axes[0].legend()
    accuracy_column = f'finetuned_accuracy_{max(FINETUNE_EPOCHS)}' if RUN_FINETUNE else 'immediate_accuracy'
    for target_sparsity in TARGET_SPARSITIES:
        plot_df = results_df[results_df['target_sparsity'] == target_sparsity]
        axes[1].plot(
            plot_df['method'], plot_df[accuracy_column],
            marker='o', label=f'{target_sparsity:.0%} sparsity'
        )
    axes[1].set_ylabel('Test accuracy (%)')
    axes[1].set_title(f'Seed {seed}: accuracy after fine-tuning')
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(output_dir / 'full_network_comparison.png', dpi=160)
    plt.close(fig)

    print(f'Seed {seed} completed in {time.monotonic() - started:.1f}s')
    return results_df

all_results = [run_seed(seed) for seed in SEEDS]
results_df = pd.concat(all_results, ignore_index=True)
results_df.to_csv(OUTPUT_ROOT / 'all_seeds_method_results.csv', index=False)

numeric_columns = [
    'test_mean_distortion', 'test_distortion_regret',
    'immediate_accuracy', 'accuracy_loss_immediate',
    'finetuned_accuracy_5', 'accuracy_loss_finetuned_5',
    'finetuned_accuracy_20', 'accuracy_loss_finetuned_20',
    'sparsity_actual', 'retained_fraction',
    'flops_reduction_fraction',
]
aggregate = results_df.groupby(['target_sparsity', 'method'])[numeric_columns].agg(['mean', 'std'])
aggregate.to_csv(OUTPUT_ROOT / 'all_seeds_mean_std.csv')
display(aggregate)
(OUTPUT_ROOT / 'all_seeds_summary.json').write_text(
    json.dumps({
        'seeds': SEEDS,
        'target_sparsities': TARGET_SPARSITIES,
        'calibration_samples': CALIB_SAMPLES,
        'test_samples': TEST_SAMPLES,
        'curve_points': CURVE_POINTS,
        'fisher_probes': FISHER_PROBES,
        'results': results_df.to_dict(orient='records'),
    }, indent=2, default=jsonable) + '\n'
)

## Publish results

Results are copied to `results/rdhp_rdp_full_network_corrected_all_seeds`.

In [ ]:
if not PUBLISH_RESULTS:
    print('Publishing disabled.')
else:
    from google.colab import userdata

    write_token = userdata.get('GITHUB_WRITE_TOKEN')
    if not write_token:
        raise RuntimeError('Add GITHUB_WRITE_TOKEN with Contents read/write access.')

    upload_repo = Path('/content/rdhp-rdp-full-network-upload')
    if upload_repo.exists():
        shutil.rmtree(upload_repo)

    askpass = Path('/tmp/github-askpass-full-network-write.sh')
    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        "  *Username*) printf '%s\\n' x-access-token ;;\n"
        "  *Password*) printf '%s\\n' \"$GITHUB_WRITE_TOKEN\" ;;\n"
        "  *) printf '\\n' ;;\n"
        'esac\n'
    )
    askpass.chmod(0o700)

    upload_env = os.environ.copy()
    upload_env.update({
        'GIT_ASKPASS': str(askpass),
        'GIT_TERMINAL_PROMPT': '0',
        'GITHUB_WRITE_TOKEN': write_token,
    })

    try:
        subprocess.run([
            'git', 'clone',
            'https://github.com/hamidrg20001379/rdhp-rdp-colab-experiment.git',
            str(upload_repo),
        ], env=upload_env, check=True)

        destination = upload_repo / 'results' / 'rdhp_rdp_full_network_corrected_all_seeds'
        shutil.copytree(OUTPUT_ROOT, destination, dirs_exist_ok=True)

        subprocess.run(['git', '-C', str(upload_repo), 'config', 'user.name', 'Colab Results Bot'], check=True)
        subprocess.run(['git', '-C', str(upload_repo), 'config', 'user.email', 'colab-results@users.noreply.github.com'], check=True)
        subprocess.run(['git', '-C', str(upload_repo), 'add', 'results', '--'])
        changed = subprocess.run([
            'git', '-C', str(upload_repo), 'diff', '--cached', '--quiet'
        ]).returncode != 0

        if changed:
            subprocess.run([
                'git', '-C', str(upload_repo), 'commit', '-m',
                'Publish full-network corrected allocation results for seeds 42-44',
            ], check=True)
            subprocess.run([
                'git', '-C', str(upload_repo), 'push', 'origin', 'HEAD:master',
            ], env=upload_env, check=True)
            print('Full-network results uploaded.')
        else:
            print('No new results to upload.')
    finally:
        askpass.unlink(missing_ok=True)